# MG Score Distributions by Text Length

Compare stored classifier `mg_score` values for human-generated (HG) and machine-generated (MG) texts across token-length bins and feature sets.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import pandas as pd

import seaborn as sns
from IPython.display import display

from notebooks.utils.mg_score_length_distributions_utils import (
    load_mg_score_length_frame,
    plot_mg_score_by_length,
)

sns.set_theme(style="whitegrid", context="notebook")
DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))

## Parameters

Each plot uses one evaluation dataset and classifier. Feature sets are shown as separate panels with shared HG/MG colors and score scale.

In [ ]:
RUN_ID = "feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100"
SCORER_LLM = "gemma_3_270m_it"
FEATURE_SETS = ("simple_similarity", "scalar_all")
MODEL_FAMILIES = ("logistic_regression", "random_forest")
TRAINING_COMPOSITION = "Kaggle+MAGE+Outfox+BBC"
EVALUATION_COMPOSITIONS = ("kaggle_train", "mage_train", "outfox_chatgpt_train")
PLOT_DATASETS = (*EVALUATION_COMPOSITIONS, "mixed")
TRAINING_WINDOW = 32
LENGTH_BIN_SIZE = 100

## Load Selected Predictions

In [ ]:
score_frame = load_mg_score_length_frame(
    DATA_ROOT,
    run_id=RUN_ID,
    scorer_llm=SCORER_LLM,
    feature_sets=FEATURE_SETS,
    training_window=TRAINING_WINDOW,
    model_families=MODEL_FAMILIES,
    training_composition=TRAINING_COMPOSITION,
    evaluation_compositions=EVALUATION_COMPOSITIONS,
)
print(f"Rows: {len(score_frame):,}")
display(score_frame.groupby(["feature_set", "dataset", "model_family", "label"], observed=True).size().rename("rows").reset_index())

## HG/MG Score Distributions

Each figure is one evaluation dataset and one classifier. Each feature set has its own panel, while HG and MG distributions use the same colors and y-axis scale.

In [ ]:
if "score_frame" not in globals():
    score_frame = load_mg_score_length_frame(
        DATA_ROOT,
        run_id=RUN_ID,
        scorer_llm=SCORER_LLM,
        feature_sets=FEATURE_SETS,
        training_window=TRAINING_WINDOW,
        model_families=MODEL_FAMILIES,
        training_composition=TRAINING_COMPOSITION,
        evaluation_compositions=EVALUATION_COMPOSITIONS,
    )

plot_frame = pd.concat(
    [score_frame, score_frame.assign(dataset="mixed")],
    ignore_index=True,
)

for dataset in PLOT_DATASETS:
    for model_family in MODEL_FAMILIES:
        figure = plot_mg_score_by_length(
            plot_frame,
            dataset=dataset,
            model_family=model_family,
            length_bin_size=LENGTH_BIN_SIZE,
        )
        display(figure)

In [ ]:
# Presentation-ready summary for the mixed evaluation data.
import matplotlib.pyplot as plt
presentation_frame = plot_frame.loc[
    plot_frame["dataset"].eq("mixed")
    & plot_frame["feature_set"].isin(("scalar_all", "simple_similarity"))
    & plot_frame["token_count"].between(100, 1199)
].copy()
presentation_frame["length_bin"] = (presentation_frame["token_count"] // 100).astype(int) * 100
presentation_frame["length_label"] = presentation_frame["length_bin"].map(lambda value: f"{value}-{value + 99}")
length_order = [f"{value}-{value + 99}" for value in range(100, 1200, 100)]
feature_specs = (("scalar_all", "Scalar all", 600), ("simple_similarity", "Simple similarity", 300))
colors = {"HG": "#2F6690", "MG": "#D1495B"}
for model_family in MODEL_FAMILIES:
    model_frame = presentation_frame.loc[presentation_frame["model_family"].eq(model_family)]
    fig, axes = plt.subplots(1, 2, figsize=(16, 6.8), sharey=True)
    for axis, (feature_set, title, cutoff) in zip(axes, feature_specs):
        feature_frame = model_frame.loc[model_frame["feature_set"].eq(feature_set)]
        sns.boxplot(data=feature_frame, x="length_label", y="mg_score", hue="label",
                    hue_order=("HG", "MG"), order=length_order, palette=colors,
                    width=0.72, linewidth=1.0, fliersize=0, ax=axis)
        cutoff_position = (cutoff - 100) // 100 - 0.5
        axis.axvspan(cutoff_position, len(length_order) - 0.5, color="#F2C14E", alpha=0.13, zorder=0)
        axis.axvline(cutoff_position, color="#B7791F", linewidth=2.2, linestyle="--", zorder=4)
        axis.text(cutoff_position + 0.12, 0.96, f"Recommended minimum\n{cutoff} tokens",
                  transform=axis.get_xaxis_transform(), ha="left", va="top",
                  fontsize=12, fontweight="bold", color="#7A4E00",
                  bbox={"boxstyle": "round,pad=0.35", "facecolor": "#FFF7D6", "edgecolor": "#E3B341"})
        axis.set_title(title, fontsize=18, fontweight="bold", pad=14)
        axis.set_xlabel("Text length (tokens)", fontsize=12, labelpad=10)
        axis.set_ylabel("MG score" if axis is axes[0] else "", fontsize=12)
        axis.set_ylim(0, 1)
        axis.tick_params(axis="x", rotation=45, labelsize=9)
        axis.grid(axis="y", color="#D9E2EC", linewidth=0.8)
        axis.grid(axis="x", visible=False)
        axis.set_axisbelow(True)
        axis.spines[["top", "right"]].set_visible(False)
        if axis is axes[1]:
            axis.get_legend().remove()
    handles, labels = axes[0].get_legend_handles_labels()
    axes[0].legend(handles, labels, title="Class", frameon=False, loc="lower left")
    fig.suptitle(f"{model_family.replace('_', ' ').title()}: more text makes the signal easier to separate", fontsize=23, fontweight="bold", y=1.02)
    fig.text(0.5, 0.965, "Mixed evaluation data | shaded area marks the usable range", ha="center", fontsize=12, color="#52606D")
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    display(fig)